# 02.5 — Reduce Datasets
Optional script to reduce ECOSoundSet and InsectSet459 down to only desired (user-selected) recordings. Helps to reduce unwanted weight form EcoSoundSet and InsectSet459 datasets.

**Kernel:** `Python (orthoptera-training)`   
**Requirements:** Send2Trash library — `pip install Send2Trash` if not satisfied.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from send2trash import send2trash

PROJECT_ROOT = Path.cwd().parent.parent  # cwd relative to current notebook path
ECO_ROOT    = PROJECT_ROOT / "datasets" / "ecosoundset"
INSECT_ROOT = PROJECT_ROOT / "datasets" / "insectset459"

# Canonical species names as they appear in ECOSoundSet (trinomial).
# InsectSet459 uses underscore format — mapped below.
# Meconema thalassinum is absent from both datasets.
UK_SPECIES = [
    "Chorthippus brunneus brunneus",          # Field Grasshopper
    "Pseudochorthippus parallelus parallelus", # Meadow Grasshopper
    "Omocestus viridulus",                    # Common Green Grasshopper
    "Tettigonia viridissima",                 # Great Green Bush-cricket
    "Roeseliana roeselii",                    # Roesel's Bush-cricket
    "Pholidoptera griseoaptera",              # Dark Bush-cricket
    "Leptophyes punctatissima",               # Speckled Bush-cricket
    "Gryllus campestris",                     # Field Cricket
]

# InsectSet459 species_name uses underscores and may be binomial.
# Map species_name from InsectSet to eco canonical name
INSECT_TO_ECO = {
    "Chorthippus_brunneus":                   "Chorthippus brunneus brunneus",
    "Pseudochorthippus_parallelus":           "Pseudochorthippus parallelus parallelus",
    "Omocestus_viridulus":                    "Omocestus viridulus",
    "Tettigonia_viridissima":                 "Tettigonia viridissima",
    "Roeseliana_roeselii":                    "Roeseliana roeselii",
    "Pholidoptera_griseoaptera":              "Pholidoptera griseoaptera",
    "Leptophyes_punctatissima":               "Leptophyes punctatissima",
    "Gryllus_campestris":                     "Gryllus campestris"
}


## EcoSoundSet Reduction

In [ ]:
# Highlight available categories to inform which recordings to keep
all_annot_eco = pd.read_csv(ECO_ROOT / "annotated_audio_segments.csv")
label_categories = all_annot_eco["label_category"].unique()
label_categories


In [ ]:
# ── Define categories to keep ──────────────────────────────────────────
keep_labels = [
    "Passeriformes","Columbiformes",
    "Strigiformes", "Geophony",
    "Accipitriformes", "Anthropophony"
]

# ── Load EcoSoundSet and remove unwanted recordings ────────────────────
eco_to_keep = all_annot_eco[
    (
        (all_annot_eco["label_category"] == "Orthoptera") &
        (all_annot_eco["label"].isin(UK_SPECIES))
    ) |
    (all_annot_eco["label_category"].isin(keep_labels))
].copy()

# Many recordings have multiple annotations, so set ensures no duplicate file names and O(1) lookup.
eco_files_to_keep = set(eco_to_keep["audio_segment_file_name"])
print(f"Total unique file names: {all_annot_eco['audio_segment_file_name'].nunique()}")
print(f"Unique files to keep: {len(eco_files_to_keep)}")

eco_files_removed = 0
eco_storage_saved = 0

for file_path in ECO_ROOT.rglob("*.wav"):
    if file_path.name not in eco_files_to_keep:
        eco_storage_saved += file_path.stat().st_size / (1024 ** 3)
        send2trash(str(file_path))
        eco_files_removed += 1

print(f"Files Removed: {eco_files_removed}\nStorage Saved: {eco_storage_saved:.2f} GB")


## InsectSet459 Reduction

In [ ]:
# ── Load InsectSet459 and remove unwanted recordings ──────────────────────────
all_annot_insect = pd.read_csv(INSECT_ROOT / "InsectSet459_Train_Val_Annotation.csv")

# Map matching species_names to canonical counterpart in INSECT_TO_ECO.
# .map() returns NaN for any key (species_name) not in INSECT_TO_ECO.
all_annot_insect["species"] = all_annot_insect["species_name"].map(INSECT_TO_ECO)

# InsectSet459 only contains Orthoptera and Cicadidae species, so there are not categories to consider keeping.
insect_to_keep = all_annot_insect[
    (all_annot_insect["group"] == "Orthoptera") & (all_annot_insect["species"].isin(UK_SPECIES))
].copy()

insect_files_to_keep = set(insect_to_keep["file_name"])
print(f"Total unique file names: {all_annot_insect['file_name'].nunique()}")
print(f"Unique files to keep: {len(insect_files_to_keep)}")

insect_files_removed = 0
insect_storage_saved = 0

for file_path in INSECT_ROOT.rglob("*"):
    if file_path.suffix in (".wav", ".mp3") and not file_path.name in insect_files_to_keep:
        insect_storage_saved += file_path.stat().st_size / (1024 ** 3)
        send2trash(str(file_path))
        insect_files_removed += 1

print(f"Files Removed: {insect_files_removed}\nStorage Saved: {insect_storage_saved:.2f} GB")
